In [2]:
import jfr_processor.profilelib.*

val project_root = "/home/Martin.Zimen/IdeaProjects/ktor/"
val profile_root = project_root + "profiler_output_weekend/"



In [3]:
val (jvm_freq, native_freq) = get_freqs(project_root, profile_root)

JVM functions to sources map loaded
Ambiguos DWARF info for 60 functions
Native functions to sources map loaded
ERROR: jvm-io.ktor.client.plugins.compression.ContentEncodingTest.testNoEncoding
ERROR: jvm-io.ktor.client.plugins.auth.AuthTest.testBasicAuthPerRealm
ERROR: jvm-io.ktor.test.RunTestWithDataTest.testSuccessAfterTimeoutItem
ERROR: jvm-io.ktor.server.config.yaml.YamlConfigTest.testEnvironmentVariable
ERROR: jvm-io.ktor.client.plugins.auth.AuthTest.testBasicAuthMultiple
ERROR: jvm-io.ktor.client.tests.plugins.WebSocketRemoteTest.testErrorHandling
ERROR: jvm-io.ktor.client.plugins.bomremover.BomRemoverTest.testNoBody
ERROR: jvm-io.ktor.client.tests.plugins.WebSocketRemoteTest.testSessionTermination
ERROR: jvm-io.ktor.server.config.yaml.YamlConfigTest.testExistingEnvironmentVariableWithDefault
ERROR: jvm-io.ktor.client.plugins.compression.ContentEncodingTest.testDeflate
ERROR: jvm-io.ktor.client.tests.plugins.WebSocketRemoteTest.testSessionClose
ERROR: jvm-io.ktor.client.plugins.a

In [4]:
data class FunctionStats(val jvm_count: Int, val native_count: Int, val jvm_src: List<Pair<String, Int>>, val native_src: List<Pair<String, Int>>)

(jvm_freq.map { Triple("JVM", it.key, it.value) } + native_freq.map { Triple("Native", it.key, it.value) })
    .filter { it.second.startsWith("kotlin.") }
    .filterNot { it.second.contains(".jvm.") }
    .groupBy { it.second.substringBefore("__at__").substringAfterLast(".") }
    .mapValues {
        fun process(list: List<Triple<String, String, Int>>): Pair<List<Pair<String, Int>>, Int> =
            list.map { it.second to it.third }
                .let { it to it.sumOf { it.second } }

        val (jvm, native) = it.value.partition { it.first == "JVM" }
        val (jvm_src, jvm_count) = process(jvm)
        val (native_src, native_count) = process(native)
        FunctionStats(jvm_count, native_count, jvm_src, native_src)
    }
    .toList()
    .filter { it.second.jvm_count > 2 && it.second.native_count > 2 }
    .sortedByDescending {  it.second.native_count.toFloat() / it.second.jvm_count }
    .forEach {
        println(it.first)
        println(it.second.native_count.toFloat() / it.second.jvm_count)
        println("JVM: ${it.second.jvm_count}")
        it.second.jvm_src.forEach { println(it) }
        println()
        println("Native: ${it.second.native_count}")
        it.second.native_src.forEach { println(it) }
        println()
        println()
    }

toByteArray
2.2
JVM: 5
(kotlin.collections.CollectionsKt___CollectionsKt.toByteArray, 5)

Native: 11
(kotlin.collections.toByteArray__at__kotlin.collections.Collection<kotlin.Byte>, 11)


first
1.8888888
JVM: 90
(kotlin.text.StringsKt___StringsKt.first, 25)
(kotlin.collections.CollectionsKt___CollectionsKt.first, 65)

Native: 170
(kotlin.text.first__at__kotlin.CharSequence, 88)
(kotlin.collections.first__at__kotlin.collections.List<0:0>, 79)
(kotlin.collections.first__at__kotlin.collections.Iterable<0:0>, 3)


component1
1.882353
JVM: 17
(kotlin.Pair.component1, 17)

Native: 32
(kotlin.Pair.component1, 32)


component2
1.625
JVM: 8
(kotlin.Pair.component2, 8)

Native: 13
(kotlin.Pair.component2, 13)


assertNotSame
1.3333334
JVM: 12
(kotlin.test.junit5.JUnit5Asserter.assertNotSame, 2)
(kotlin.test.AssertionsKt__AssertionsKt.assertNotSame, 5)
(kotlin.test.AssertionsKt.assertNotSame, 5)

Native: 16
(kotlin.test.Asserter.assertNotSame, 9)
(kotlin.test.assertNotSame, 7)


drop
1.2777778
JV

In [8]:
val native_debug_map = ktor_get_native_debug_map(project_root)
val jvm_fun_to_src_map = ktor_get_fun_to_src_map(project_root)

Ambiguos DWARF info for 60 functions


In [11]:
get_valid_tests(profile_root).forEach { testname ->
    lazy_samples_new(profile_root + "jvm-" + testname + ".jfr")
        .mapNotNull { truncate_stack_trace(it) { jvm_get_name(it) == testname } }
        .filter { it.any {
            jvm_get_name(it)
                ?.substringAfterLast(".")
                ?.let { it == "toByteArray" }
                ?: false
                    && jvm_get_name(it)?.startsWith("kotlin.") ?: false
        }}
        .filter {
            it.all {
                jvm_get_name(it)
                    ?.let { jvm_fun_to_src_map[it] }
                    ?.all { it.contains("/common/") }
                    ?: true
            }
        }
        .forEach {
            println("JVM stack trace")
            it.forEach { println(jvm_get_name(it)) }
            println()
        }
}

ERROR: jvm-io.ktor.client.plugins.compression.ContentEncodingTest.testNoEncoding
ERROR: jvm-io.ktor.client.plugins.auth.AuthTest.testBasicAuthPerRealm
ERROR: jvm-io.ktor.test.RunTestWithDataTest.testSuccessAfterTimeoutItem
ERROR: jvm-io.ktor.server.config.yaml.YamlConfigTest.testEnvironmentVariable
ERROR: jvm-io.ktor.client.plugins.auth.AuthTest.testBasicAuthMultiple
ERROR: jvm-io.ktor.client.tests.plugins.WebSocketRemoteTest.testErrorHandling
ERROR: jvm-io.ktor.client.plugins.bomremover.BomRemoverTest.testNoBody
ERROR: jvm-io.ktor.client.tests.plugins.WebSocketRemoteTest.testSessionTermination
ERROR: jvm-io.ktor.server.config.yaml.YamlConfigTest.testExistingEnvironmentVariableWithDefault
ERROR: jvm-io.ktor.client.plugins.compression.ContentEncodingTest.testDeflate
ERROR: jvm-io.ktor.client.tests.plugins.WebSocketRemoteTest.testSessionClose
ERROR: jvm-io.ktor.client.plugins.auth.AuthTest.testDigestAuthSHA256
ERROR: jvm-io.ktor.client.plugins.auth.AuthTest.testUnauthorizedDoesNotRefresh

In [11]:
jvm_fun_to_src_map["io.ktor.server.config.yaml.YamlConfigTest.testKeysTopLevelYamlConfig"]?.any { !it.contains("/common/") }

true

In [14]:
get_valid_tests(profile_root).flatMap { testname ->
    lazy_samples_new(profile_root + "linuxX64-" + testname + ".jfr")
        .mapNotNull { truncate_stack_trace(it) { native_get_name(it)?.contains(testname) ?: false } }
        .filter {
            it.any {
//                native_debug_map[it.method.name]
//                    ?.name
//                    ?.let { it.also { println(it) } == "collectionSizeOrDefault" }
//                    ?: false
                native_get_name(it)?.contains("collectionSizeOrDefault") ?: false
            }
        }
        .filter {
            it.all {
                native_debug_map[it.method.name]?.dir?.contains("/common/") ?: true
            }
        }
        .map { it.last() }
        .toList()
//        .groupBy { it }
//        .mapValues { it.value.size }
//        .let {println(it) }
//        .forEach {
//            println("Native stack trace")
//            it.forEach { println(native_get_name(it)) }
//            println()
//        }
}.groupBy { it.method.name }.mapValues { it.value.size }.let { println(it) }

ERROR: jvm-io.ktor.client.plugins.compression.ContentEncodingTest.testNoEncoding
ERROR: jvm-io.ktor.client.plugins.auth.AuthTest.testBasicAuthPerRealm
ERROR: jvm-io.ktor.test.RunTestWithDataTest.testSuccessAfterTimeoutItem
ERROR: jvm-io.ktor.server.config.yaml.YamlConfigTest.testEnvironmentVariable
ERROR: jvm-io.ktor.client.plugins.auth.AuthTest.testBasicAuthMultiple
ERROR: jvm-io.ktor.client.tests.plugins.WebSocketRemoteTest.testErrorHandling
ERROR: jvm-io.ktor.client.plugins.bomremover.BomRemoverTest.testNoBody
ERROR: jvm-io.ktor.client.tests.plugins.WebSocketRemoteTest.testSessionTermination
ERROR: jvm-io.ktor.server.config.yaml.YamlConfigTest.testExistingEnvironmentVariableWithDefault
ERROR: jvm-io.ktor.client.plugins.compression.ContentEncodingTest.testDeflate
ERROR: jvm-io.ktor.client.tests.plugins.WebSocketRemoteTest.testSessionClose
ERROR: jvm-io.ktor.client.plugins.auth.AuthTest.testDigestAuthSHA256
ERROR: jvm-io.ktor.client.plugins.auth.AuthTest.testUnauthorizedDoesNotRefresh